### K-Nearest Neighbours

In [1]:
from scipy.sparse import load_npz
import numpy as np
from sklearn.preprocessing import normalize

In [2]:
# 1) Cargamos las matrices de usuarios-items, tanto de train como de test: 
path = '../data/processed/'

train_set = load_npz(path+'train_set.npz')
test_set = load_npz(path+'test_set.npz')

In [3]:
# 2) Establecemos una forma de calcular similaridad: 

# Debido a nuestro uso pasado de la similaridad coseno en PLN y sus buenos resultados que nos dio, 
# elegiremos éste mismo (calculado como sim_cos = (u * v) / (||u|| * ||v||), siendo u y v los usuarios): 

def cosine_similarity(matrix):
    """
    Calcula la similitud coseno entre todas las filas de una matriz CSR.
    Retorna una matriz de similitud (usuarios x usuarios) en formato CSR.
    """

    # 1) Calculamos el producto escalar entre cada usuario por cada usuario, lo cual 
    # matricialmente se consigue de forma eficiente mediante la multiplicación de la matriz por su traspuesta: 
    dot_product = matrix @ matrix.T # Matriz de Gram. 
    
    # 2) Calculamos la norma de cada vector fila: 
    norms = np.sqrt(np.array(matrix.power(2).sum(axis=1)).flatten())
    
    # 3) Evitamos división por cero (usuarios sin ninguna escucha): 
    norms[norms == 0] = 1.0
    
    # 4) Aplicar la normalización: dot_product / (norm_i * norm_j)
    # Esto es equivalente a: (1/norm) * dot_product * (1/norm), donde 1/norm son matrices diagonales. 
    inv_norms = 1.0 / norms
    similarity = dot_product.multiply(inv_norms[:, np.newaxis]).multiply(inv_norms[np.newaxis, :]) # Usamos broadcasting mediante np.newaxis 
                                                                                                   # para imitar el comportamiento de una diagonal
                                                                                                   # de forma eficiente.  
    
    return similarity.tocsr()

# No obstante, observamos que es costosa en tiempo y memoria. 

In [ ]:
# 2.1) Versión optimizada de similaridad coseno: 

def cosine_similarity_optimized(matrix):
    """
    Calcula la similitud coseno entre las filas de una matriz CSR.
    Aplica normalización L2 previa para maximizar la eficiencia.
    """
    
    # 1) Normalizamos cada fila de la matriz original dividiéndola por su norma L2.
    # Esto transforma cada vector de usuario en un vector unitario (de longitud 1).
    matrix_normalized = normalize(matrix, norm='l2', axis=1)
    
    # 2) Al estar ya normalizados, el producto escalar es exactamente la similitud coseno.
    # Esta operación (C = A @ A.T) la resuelve SciPy internamente en C++.
    similarity = matrix_normalized @ matrix_normalized.T
    
    return similarity

# Comprobación del funcionamiento: 
sim_matrix = cosine_similarity_optimized(train_set)
sim_matrix